<a href="https://colab.research.google.com/github/COL-O/POTOK/blob/main/notebooks/potok-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/COL-O/POTOK/blob/main/notebooks/potok-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# POTOK Colab: полная верификация `main` (вне Actions)

**Схема та же, что и раньше, плюс доказанный обход занятых портов.** Одна тетрадь — полный круг проверок `main`:

1. свежий клон `main` →
2. Go 1.22.12 официальным архивом →
3. `go test -race -count=1 ./...` →
4. 5× бывший флейк фронта (`TestRouterRoutineVetoAndQuarantine`) →
5. `go build` + `examples/simple` + `examples/drone` →
6. организм 4 такта — **через `unshare -n`** (в Colab порты `:8080`/`:9000` заняты инфраструктурой: `node` pid 6, `jupyter-server` pid 120 — их не трогаем) →
7. автоматическая сверка «распечатка ↔ метрики» (КОН 2).

Эталон на `main ≥ 832e3d0` (после PR #24): все тесты `ok`; флейк 5/5 PASS; организм rc 0; такт 3 — **Gen#2 Guard=[-5.1..31.2], BlackSwans=1/1, Alpha=0.1**; тепловой отчёт `Новых законов=1 | Реабилитаций Veto=1 | WAL=1` — распечатка и метрики сходятся.

⚠️ Секретов нет; тетрадь никуда ничего не отправляет; записывает только в `/content`. Итоги — в журналы дома (LOGBOOK/RUN-LOG) руками человека.

In [3]:
# 1. Свежий клон main
import os, subprocess
REPO = '/content/potok-run'
subprocess.run(['rm', '-rf', REPO], check=False)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
                'https://github.com/COL-O/POTOK.git', REPO], check=True)
print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)


975df60 Merge pull request #25 from COL-O/arena/3ce48adc-potok



In [4]:
# 2. Go 1.22.12 официальным архивом
import os, subprocess
subprocess.run('curl -sL https://dl.google.com/go/go1.22.12.linux-amd64.tar.gz | tar -C /usr/local -xz',
               shell=True, check=True)
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
print(subprocess.run(['go', 'version'], capture_output=True, text=True).stdout)


go version go1.22.12 linux/amd64



In [5]:
# 3. Иммунитет: полный прогон с гонками (как шаг CI)
import os, subprocess
REPO = '/content/potok-run'
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
r = subprocess.run(['go', 'test', '-race', '-count=1', './...'],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout[-4000:])
if r.stderr:
    print(r.stderr[-1500:])
print('ВЕРДИКТ ТЕСТОВ:', 'PASS' if r.returncode == 0 else f'FAIL (код {r.returncode})')


ok  	organism	1.024s
?   	organism/contracts	[no test files]
?   	organism/core	[no test files]
?   	organism/examples/drone	[no test files]
?   	organism/examples/simple	[no test files]
?   	organism/links	[no test files]
?   	organism/modules/bim	[no test files]
ok  	organism/cmd/organism	1.054s
ok  	organism/modules/bus	1.030s
--- FAIL: TestRouterZeroAllocations (0.01s)
    router_test.go:136: unexpected step failure: err=<nil>, v={OrganID:6 Status:4 Confidence:0 ActionCode:214 SuggestedMask:0 ResultOffset:0}
FAIL
FAIL	organism/modules/front	0.028s
ok  	organism/modules/graph	1.024s
?   	organism/modules/growth/projector	[no test files]
ok  	organism/modules/growth	1.018s
ok  	organism/modules/network	1.026s
ok  	organism/modules/polygon	1.020s
?   	organism/modules/store	[no test files]
?   	organism/modules/store/memory	[no test files]
ok  	organism/modules/seed	2.320s
ok  	organism/modules/stratos	1.020s
FAIL

go: downloading golang.org/x/sys v0.27.0

ВЕРДИКТ ТЕСТОВ: FAIL (код 1)

In [6]:
# 4. Бывший флейк фронта: 5 подряд
import os, subprocess
REPO = '/content/potok-run'
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
for i in range(1, 6):
    r = subprocess.run(['go', 'test', '-race', '-count=1',
                        '-run', 'TestRouterRoutineVetoAndQuarantine', './modules/front/'],
                       cwd=REPO, capture_output=True, text=True)
    print(f'флейк-прогон {i}:', 'PASS' if r.returncode == 0 else 'FAIL')


флейк-прогон 1: PASS
флейк-прогон 2: PASS
флейк-прогон 3: PASS
флейк-прогон 4: PASS
флейк-прогон 5: PASS


In [7]:
# 5. Сборка и два примера
import os, subprocess
REPO = '/content/potok-run'
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
subprocess.run(['go', 'build', '-o', 'bin/organism', './cmd/organism/...'], cwd=REPO, check=True)
for name in ('examples/simple', 'examples/drone'):
    print('---', name)
    r = subprocess.run(['go', 'run', './' + name + '/main.go'], cwd=REPO,
                       capture_output=True, text=True, timeout=300)
    print(r.stdout or '(пусто)')
    if r.stderr:
        print(r.stderr[-1000:])
    print('код возврата:', r.returncode)


--- examples/simple
Знакомый импульс: Status=1, Action=1, Confidence=1.00
Неизвестный импульс: Status=3, Surprises=1

код возврата: 0
--- examples/drone
=== ЗАПУСК ПРОФИЛЯ АВТОНОМНОГО ДРОНА (КОН 8: ДЕЛЕНИЕ) ===
[Полёт 1] Штатный режим: Status=1, Confidence=1.00, Вызовов DroneAcoustic=1
[Полёт 2] Акустический удар: ActionCode=707 (Уклонение), Confidence=0.96, NextMask=0x30

код возврата: 0


In [8]:
# 6. Организм: 4 такта в изолированном netns (обход занятого Colab'ом :8080/:9000)
import os, subprocess
REPO = '/content/potok-run'
r = subprocess.run(['unshare', '-n', REPO + '/bin/organism'], cwd=REPO,
                   capture_output=True, text=True, timeout=180)
print(r.stdout)
if r.stderr:
    print('--- stderr ---')
    print(r.stderr)
print('КОД ВОЗВРАТА ОРГАНИЗМА:', r.returncode)


=== ИНИЦИАЛИЗАЦИЯ ЦИФРОВОГО ОРГАНИЗМА (4 КОНТУРА ВРЕМЕНИ) ===
[Dashboard] HTTP-сервер прозрачности готов на :8080 (/health, /metrics, /prometheus)
[Gateway]   UDP-рецептор телеметрии и графов запущен на :9000
[Такт 1 | front]   Штатный рейс: Status=1, Confidence=1.00 | UDP-пакетов=0
[Такт 2 | front]   Слепое пятно HDC: Status=3 -> взведён Surprise!
[Такт 2 | polygon] Диагноз поставлен: Cause=77, Antibody=888, Confidence=0.91
[Такт 3 | polygon] Выход за паспорт KAN (25.2 > 10.0) -> Эскалация в growth!
[Такт 3 | growth]  Выведен новый закон Gen#2: Guard=[-5.1..31.2], BlackSwans=1/1, Alpha=0.1
[Такт 4 | front]   Повтор сложного сбоя: отработан как рефлекс! Confidence=1.00

=== ИТОГОВЫЙ ТЕПЛОВОЙ ОТЧЁТ ОРГАНИЗМА (КОН 10) ===
Шина (Bus):     Опубликовано=4 | Ошибок CRC32=0 | Пропусков эпохи=0
Слой front:     Surprise=1 | Заблокировано Veto=0 | Эскалаций в growth=0
Слой polygon:   Диагнозов=1 | Антител в Карантин=1 | Выдано Veto=0
Слой growth:    Новых законов=1 | Забраковано=0 | Реабилитаций

In [9]:
# 7. Сверка распечатки и метрик (КОН 2): закон выведен — значит, он и в учёте
import os, subprocess
REPO = '/content/potok-run'
r = subprocess.run(['unshare', '-n', REPO + '/bin/organism'], cwd=REPO,
                   capture_output=True, text=True, timeout=180)
out = r.stdout
print(out)
import re
m_laws = re.search(r'Новых законов=(\d+)', out)
m_wal  = re.search(r'WAL=(\d+)', out)
m_claim = 'Выведен новый закон' in out
laws = int(m_laws.group(1)) if m_laws else -1
wal  = int(m_wal.group(1)) if m_wal else -1
print(f'распечатка: «Выведен новый закон» = {m_claim}; метрики: Новых законов={laws}, WAL={wal}')
same = (m_claim == (laws > 0)) and (laws == wal)
print('СВЁРКА:', 'СОГЛАСНО — распечатка и метрики сходятся' if same else 'РАСХОЖДЕНИЕ — факт для задачи №6')
print('ИТОГ PR-24/MAIN:', 'ЗЕЛЁНО' if (same and r.returncode == 0) else 'КРАСНО')


=== ИНИЦИАЛИЗАЦИЯ ЦИФРОВОГО ОРГАНИЗМА (4 КОНТУРА ВРЕМЕНИ) ===
[Dashboard] HTTP-сервер прозрачности готов на :8080 (/health, /metrics, /prometheus)
[Gateway]   UDP-рецептор телеметрии и графов запущен на :9000
[Такт 1 | front]   Штатный рейс: Status=1, Confidence=1.00 | UDP-пакетов=0
[Такт 2 | front]   Слепое пятно HDC: Status=3 -> взведён Surprise!
[Такт 2 | polygon] Диагноз поставлен: Cause=77, Antibody=888, Confidence=0.91
[Такт 3 | polygon] Выход за паспорт KAN (25.2 > 10.0) -> Эскалация в growth!
[Такт 3 | growth]  Выведен новый закон Gen#2: Guard=[-5.1..31.2], BlackSwans=1/1, Alpha=0.1
[Такт 4 | front]   Повтор сложного сбоя: отработан как рефлекс! Confidence=1.00

=== ИТОГОВЫЙ ТЕПЛОВОЙ ОТЧЁТ ОРГАНИЗМА (КОН 10) ===
Шина (Bus):     Опубликовано=4 | Ошибок CRC32=0 | Пропусков эпохи=0
Слой front:     Surprise=1 | Заблокировано Veto=0 | Эскалаций в growth=0
Слой polygon:   Диагнозов=1 | Антител в Карантин=1 | Выдано Veto=0
Слой growth:    Новых законов=1 | Забраковано=0 | Реабилитаций

## Полная сборка (пуско-наладка) — шаги 8–11

Мяч дальше: не только «код честен», но и «организм живёт как сервис». Всё внутри одной изолированной сети (`unshare -n`) — порты Colab не занимаем и не освобождаем, ничего не трогаем.

Шаги 9 и 10 — информационные (могут краснить по невинным причинам: сеть `go mod tidy`, иначе — факт для разбора). Шаг 8 — ключевой: демон, `/health`, пульс шины через два Tensor-кадра (`magic 0x4F524731`, UDP :9000).

**Ожидается (эталон песочницы, прогон 0.2):** `/health` → `{"status":"healthy","published_ticks":N}`; второй снимок `/metrics` покажет `Published` на 2 больше и `Surprise` +1.

In [ ]:
# 8. Демон: /health + пульс шины через 2 Tensor-кадра (UDP :9000)
import subprocess
FULL = r'''
ORGANISM_DAEMON=1 /content/potok-run/bin/organism &
ORG=$!
sleep 2
echo "=== /health"
curl -s http://127.0.0.1:8080/health; echo
echo "=== /metrics (до кадров)"
curl -s http://127.0.0.1:8080/metrics; echo
python3 - <<'PY'
import socket, struct, time
MAGIC_T, TTS, PRIORITY = 0x4F524731, 150, 10
def frame(vals):
    hdr = struct.pack('<IIIBB', MAGIC_T, 0, TTS, PRIORITY, len(vals))
    hdr = hdr + b'\x00' * (24 - len(hdr))
    return hdr + struct.pack('<%df' % len(vals), *vals)
for _ in range(2):
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    s.sendto(frame([7.5, 8.0, 8.2, 8.5]), ('127.0.0.1', 9000))
    s.close()
    time.sleep(0.5)
print('>> 2 кадра Tensor отправлено в :9000')
PY
sleep 1
echo "=== /metrics (после кадров)"
curl -s http://127.0.0.1:8080/metrics; echo
kill -INT $ORG  # SIGINT: контур сам закрывается (graceful)
wait $ORG 2>/dev/null || true
'''
r = subprocess.run(['unshare', '-n', 'bash', '-c', FULL],
                   capture_output=True, text=True, timeout=90)
print(r.stdout)
if r.stderr:
    print('--- stderr ---')
    print(r.stderr[-2500:])
print('итог ячейки:', 'OK' if '/health' in r.stdout else 'СМОТРЕТЬ ВЫШЕ')


In [ ]:
# 9. flat/ --check — плоский контур (информационно)
import subprocess
r = subprocess.run('bash bootstrap.sh --check', shell=True,
                   cwd='/content/potok-run/flat',
                   capture_output=True, text=True, timeout=900)
print(r.stdout[-2500:])
if r.stderr:
    print(r.stderr[-1000:])
print('flat --check код возврата:', r.returncode)


In [ ]:
# 10. Корневой bootstrap.sh --check — каркас и окружение (информационно)
import subprocess
r = subprocess.run('bash bootstrap.sh --check', shell=True,
                   cwd='/content/potok-run',
                   capture_output=True, text=True, timeout=600)
print(r.stdout[-2500:])
if r.stderr:
    print(r.stderr[-1000:])
print('bootstrap --check код возврата:', r.returncode)


## 11. Сводка полной сборки — заполняет человек

| Узел | Критерий | Факт (впишите) |
|---|---|---|
| Тесты `-race ./...` | PASS | |
| Флейк ×5 | 5/5 PASS | |
| Организм rc=0 | 4 такта, честный Gen#2 | |
| Сверка распечатки | СОГЛАСНО | |
| Демон: /health | healthy | |
| Пульс: Published Δ | +2 | |
| flat/ --check | зелёный или честное красное | |
| bootstrap --check | код 0 | |

Строка сдачи (по `community/ACCEPTANCE-FORMAT.md`): `принято: YYYY-MM-DD, ИТОГ: …` — в №12.

## Куда сообщать

Вывод ячеек → репликой в №12 (через мост: `inbox/<имя>.md` + push) или строки `ВЕРДИКТ ТЕСТОВ`, `СВЁРКА`, `ИТОГ PR-24/MAIN` — задачи №6/#12. Если Colab занят своими портами — шаг 6 уже это учитывает; ничего убивать не надо.

*Протокол верификации `main` после состыковки PR #24; зафиксирован основателем 2026-11-20. Эталон неизменности: CI main зелёный (run 44788285312), CodeQL pass.*